# Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis for the Diabetic Nephropathy prediction dataset.

## Objectives:
- Understand dataset structure and characteristics
- Identify missing values and data quality issues
- Analyze class distribution
- Explore feature distributions and relationships
- Identify patterns and correlations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Create output directory for plots
PLOTS_DIR = Path.cwd().parent / 'outputs' / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully")
print(f"Plots will be saved to: {PLOTS_DIR}")

## 1. Load Dataset

Load the dataset from the Excel file and display basic information.

In [ ]:
# Update this path to your actual dataset file
DATASET_PATH = '../dataset/diabetic_nephropathy_data.xlsx'

# Load dataset
df = pd.read_excel(DATASET_PATH)

print(f"Dataset Shape: {df.shape}")
print(f"\nDataset Info:")
print(df.info())
print(f"\nFirst 5 rows:")
display(df.head())

## 2. Dataset Overview

This visualization provides a comprehensive overview of the dataset including:
- Number of rows and columns
- Data types of each column
- Memory usage
- Basic statistics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

# Plot 1: Dataset shape
axes[0, 0].bar(['Rows', 'Columns'], [df.shape[0], df.shape[1]], 
              color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0, 0].set_title('Dataset Dimensions')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate([df.shape[0], df.shape[1]]):
    axes[0, 0].text(i, v + (df.shape[0] * 0.02), str(v), 
                   ha='center', fontweight='bold')

# Plot 2: Data types distribution
dtype_counts = df.dtypes.value_counts()
axes[0, 1].pie(dtype_counts.values, labels=dtype_counts.index, 
               autopct='%1.1f%%', startangle=90, 
               colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
axes[0, 1].set_title('Data Types Distribution')

# Plot 3: Missing values per column
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
if len(missing_values) > 0:
    axes[1, 0].barh(range(len(missing_values)), missing_values.values, 
                    color='#e74c3c', alpha=0.7)
    axes[1, 0].set_yticks(range(len(missing_values)))
    axes[1, 0].set_yticklabels(missing_values.index)
    axes[1, 0].set_xlabel('Missing Values Count')
    axes[1, 0].set_title('Missing Values by Column')
    axes[1, 0].invert_yaxis()
else:
    axes[1, 0].text(0.5, 0.5, 'No Missing Values', 
                   ha='center', va='center', fontsize=14, 
                   transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Missing Values by Column')

# Plot 4: Memory usage
memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
axes[1, 1].bar(['Memory Usage'], [memory_mb], color='#2ecc71', alpha=0.7)
axes[1, 1].set_title(f'Memory Usage: {memory_mb:.2f} MB')
axes[1, 1].set_ylabel('Megabytes (MB)')
axes[1, 1].text(0, memory_mb + (memory_mb * 0.05), f'{memory_mb:.2f} MB', 
               ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_dataset_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("Dataset overview plot saved successfully")

## 3. Missing Value Heatmap

This heatmap visualizes the pattern of missing values across the dataset.
- **Yellow/White areas**: Indicate missing values
- **Purple/Dark areas**: Indicate present values
- Helps identify if missing values are randomly distributed or clustered in specific columns

In [ ]:
# Create missing value heatmap
plt.figure(figsize=(14, 8))

# Create binary matrix of missing values
missing_matrix = df.isnull()

# Plot heatmap
sns.heatmap(missing_matrix, cbar=False, cmap='viridis', 
            yticklabels=False, cbar_kws={'label': 'Missing'})

plt.title('Missing Value Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Columns', fontsize=12)
plt.ylabel('Rows', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add statistics
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
missing_pct = (total_missing / total_cells) * 100

plt.figtext(0.5, 0.02, 
            f'Total Missing: {total_missing} ({missing_pct:.2f}%)', 
            ha='center', fontsize=12, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_missing_value_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Missing value heatmap saved. Total missing: {total_missing} ({missing_pct:.2f}%)")

## 4. Class Distribution

This visualization shows the distribution of the target variable (diabetic nephropathy classes).
- **Bar chart**: Shows absolute count of each class
- **Pie chart**: Shows percentage distribution
- Important for identifying class imbalance issues

In [ ]:
# Update this to your actual target column name
TARGET_COLUMN = 'diabetic_nephropathy'  # Change this to your target column

if TARGET_COLUMN in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Class Distribution: {TARGET_COLUMN}', 
                 fontsize=16, fontweight='bold')
    
    # Get class counts
    class_counts = df[TARGET_COLUMN].value_counts().sort_index()
    class_pct = (class_counts / len(df)) * 100
    
    # Plot 1: Bar chart
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    bars = axes[0].bar(class_counts.index, class_counts.values, 
                      color=colors[:len(class_counts)], alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Class', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Absolute Count')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add count labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Pie chart
    axes[1].pie(class_counts.values, labels=class_counts.index,
               autopct='%1.1f%%', startangle=90,
               colors=colors[:len(class_counts)],
               explode=[0.05] * len(class_counts))
    axes[1].set_title('Percentage Distribution')
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '03_class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Class distribution plot saved")
    print(f"\nClass counts:\n{class_counts}")
    print(f"\nClass percentages:\n{class_pct}")
else:
    print(f"Warning: Target column '{TARGET_COLUMN}' not found in dataset.")
    print("Please update the TARGET_COLUMN variable with your actual target column name.")

## 5. Histograms

Histograms show the distribution of numerical features.
- Helps identify skewness, outliers, and distribution patterns
- Normal distribution: Bell-shaped curve
- Skewed distribution: Asymmetric shape
- Multiple peaks: Indicates subgroups in data

In [ ]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    # Calculate number of rows and columns for subplots
    n_cols = 4
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    fig.suptitle('Feature Distributions (Histograms)', 
                 fontsize=16, fontweight='bold')
    
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            axes[idx].hist(df[col].dropna(), bins=30, color='#3498db', 
                          alpha=0.7, edgecolor='black')
            axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
            axes[idx].set_xlabel('Value')
            axes[idx].set_ylabel('Frequency')
            axes[idx].grid(axis='y', alpha=0.3)
            
            # Add statistics
            mean_val = df[col].mean()
            median_val = df[col].median()
            axes[idx].axvline(mean_val, color='red', linestyle='--', 
                           linewidth=2, label=f'Mean: {mean_val:.2f}')
            axes[idx].axvline(median_val, color='green', linestyle='--', 
                           linewidth=2, label=f'Median: {median_val:.2f}')
            axes[idx].legend(fontsize=8)
    
    # Hide empty subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '04_histograms.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Histograms saved for {len(numerical_cols)} numerical features")
else:
    print("No numerical columns found in dataset")

## 6. Boxplots

Boxplots display the distribution of numerical features through quartiles.
- **Box**: Represents interquartile range (IQR) - middle 50% of data
- **Line inside box**: Median value
- **Whiskers**: Extend to 1.5 * IQR (typical range)
- **Points outside whiskers**: Potential outliers
- Excellent for identifying outliers and comparing distributions

In [ ]:
if numerical_cols:
    # Calculate number of rows and columns for subplots
    n_cols = 4
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    fig.suptitle('Feature Distributions (Boxplots)', 
                 fontsize=16, fontweight='bold')
    
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            bp = axes[idx].boxplot(df[col].dropna(), 
                                  patch_artist=True,
                                  boxprops=dict(facecolor='#3498db', alpha=0.7),
                                  medianprops=dict(color='red', linewidth=2),
                                  whiskerprops=dict(color='black', linewidth=1.5),
                                  capprops=dict(color='black', linewidth=1.5))
            axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
            axes[idx].set_ylabel('Value')
            axes[idx].grid(axis='y', alpha=0.3)
            
            # Add statistics text
            stats = df[col].describe()
            stats_text = f'Median: {stats["50%"]:.2f}\nQ1: {stats["25%"]:.2f}\nQ3: {stats["75%"]:.2f}'
            axes[idx].text(0.95, 0.95, stats_text,
                          transform=axes[idx].transAxes,
                          fontsize=8,
                          verticalalignment='top',
                          horizontalalignment='right',
                          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Hide empty subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '05_boxplots.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Boxplots saved for {len(numerical_cols)} numerical features")
else:
    print("No numerical columns found in dataset")

## 7. Correlation Heatmap

This heatmap shows the correlation between numerical features.
- **Values range from -1 to +1**:
  - +1: Perfect positive correlation (as one increases, other increases)
  - 0: No correlation
--1: Perfect negative correlation (as one increases, other decreases)
- **Color coding**:
  - Red: Positive correlation
  - Blue: Negative correlation
  - White: No correlation
- Helps identify multicollinearity and feature relationships

In [ ]:
if numerical_cols:
    # Calculate correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    # Create heatmap
    plt.figure(figsize=(14, 12))
    
    # Use a diverging colormap (red for positive, blue for negative)
    sns.heatmap(correlation_matrix, 
                annot=True,  # Show correlation values
                cmap='RdBu_r',  # Red-Blue reversed colormap
                center=0,  # Center colormap at 0
                square=True,  # Square cells
                linewidths=0.5,  # Add grid lines
                cbar_kws={'label': 'Correlation Coefficient'},
                fmt='.2f',  # Format to 2 decimal places
                annot_kws={'size': 8})  # Annotation font size
    
    plt.title('Correlation Heatmap of Numerical Features', 
             fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '06_correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Correlation heatmap saved")
    
    # Print highly correlated pairs
    print("\nHighly correlated feature pairs (|correlation| > 0.7):")
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                high_corr_pairs.append((correlation_matrix.columns[i], 
                                       correlation_matrix.columns[j], 
                                       corr_val))
    
    if high_corr_pairs:
        for pair in high_corr_pairs:
            print(f"  {pair[0]} - {pair[1]}: {pair[2]:.3f}")
    else:
        print("  No highly correlated pairs found.")
else:
    print("No numerical columns found for correlation analysis")

## 8. Pairplot (Important Features Only)

Pairplot shows pairwise relationships between selected features.
- **Diagonal**: Distribution of each feature (histogram/KDE)
- **Off-diagonal**: Scatter plots showing relationships between feature pairs
- **Color by target**: Different colors for different classes (if target specified)
- Helps visualize complex relationships and identify patterns
- Limited to important features to avoid overcrowding

In [ ]:
# Select top features based on variance or correlation with target
# You can customize this selection based on domain knowledge

if numerical_cols and len(numerical_cols) > 1:
    # Select top 6 features with highest variance (can be customized)
    feature_variance = df[numerical_cols].var().sort_values(ascending=False)
    top_features = feature_variance.head(6).index.tolist()
    
    print(f"Selected features for pairplot: {top_features}")
    
    # Create pairplot
    if TARGET_COLUMN in df.columns and TARGET_COLUMN not in top_features:
        # Include target column for coloring
        plot_data = df[top_features + [TARGET_COLUMN]].copy()
        pairplot = sns.pairplot(plot_data, hue=TARGET_COLUMN, 
                              palette='Set2', 
                              diag_kind='hist',
                              plot_kws={'alpha': 0.6},
                              height=2.5)
    else:
        # No target column coloring
        plot_data = df[top_features].copy()
        pairplot = sns.pairplot(plot_data, 
                              diag_kind='hist',
                              plot_kws={'alpha': 0.6},
                              height=2.5)
    
    pairplot.fig.suptitle('Pairplot of Important Features', 
                         fontsize=16, fontweight='bold', y=1.02)
    
    plt.savefig(PLOTS_DIR / '07_pairplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Pairplot saved successfully")
else:
    print("Insufficient numerical columns for pairplot")

## 9. Feature Distributions by Class

This visualization shows how each feature is distributed across different classes.
- **Violin plots**: Combine boxplot and kernel density plot
- **Shows**: Distribution shape, median, quartiles for each class
- **Purpose**: Identify features that differ significantly between classes
- **Useful for**: Feature selection and understanding class separation

In [ ]:
if TARGET_COLUMN in df.columns and numerical_cols:
    # Select top features for visualization (to avoid overcrowding)
    top_n_features = min(6, len(numerical_cols))
    feature_variance = df[numerical_cols].var().sort_values(ascending=False)
    selected_features = feature_variance.head(top_n_features).index.tolist()
    
    # Calculate number of rows and columns
    n_cols = 2
    n_rows = (len(selected_features) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    fig.suptitle(f'Feature Distributions by {TARGET_COLUMN}', 
                 fontsize=16, fontweight='bold')
    
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()
    
    for idx, feature in enumerate(selected_features):
        if idx < len(axes):
            # Create violin plot
            sns.violinplot(data=df, x=TARGET_COLUMN, y=feature, 
                          ax=axes[idx], palette='Set2')
            axes[idx].set_title(feature, fontsize=10, fontweight='bold')
            axes[idx].set_xlabel('Class')
            axes[idx].set_ylabel('Value')
            axes[idx].grid(axis='y', alpha=0.3)
    
    # Hide empty subplots
    for idx in range(len(selected_features), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '08_feature_distributions_by_class.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Feature distributions by class saved for {len(selected_features)} features")
else:
    if TARGET_COLUMN not in df.columns:
        print(f"Target column '{TARGET_COLUMN}' not found")
    else:
        print("No numerical columns found")

## 10. Summary Statistics

Display comprehensive summary statistics for all numerical features.

In [ ]:
if numerical_cols:
    print("="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    # Numerical features statistics
    print("\nNumerical Features:")
    display(df[numerical_cols].describe().transpose())
    
    # Categorical features statistics
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        print("\nCategorical Features:")
        for col in categorical_cols:
            print(f"\n{col}:")
            print(df[col].value_counts())
    
    # Skewness and Kurtosis
    print("\nSkewness and Kurtosis:")
    skewness = df[numerical_cols].skew()
    kurtosis = df[numerical_cols].kurtosis()
    stats_df = pd.DataFrame({'Skewness': skewness, 'Kurtosis': kurtosis})
    display(stats_df)
    
    print("\nInterpretation:")
    print("- Skewness > 1: Highly positively skewed (right-skewed)")
    print("- Skewness < -1: Highly negatively skewed (left-skewed)")
    print("- Kurtosis > 3: Heavy-tailed distribution (more outliers)")
    print("- Kurtosis < 3: Light-tailed distribution (fewer outliers)")
else:
    print("No numerical columns found for summary statistics")

## EDA Complete

All visualizations have been saved to the `outputs/plots/` directory:

1. **01_dataset_overview.png** - Overall dataset statistics
2. **02_missing_value_heatmap.png** - Pattern of missing values
3. **03_class_distribution.png** - Target class distribution
4. **04_histograms.png** - Feature distributions
5. **05_boxplots.png** - Boxplots for outlier detection
6. **06_correlation_heatmap.png** - Feature correlations
7. **07_pairplot.png** - Pairwise relationships
8. **08_feature_distributions_by_class.png** - Distributions by target class

### Key Insights to Look For:
- Missing value patterns
- Class imbalance issues
- Skewed distributions
- Outliers in features
- Highly correlated features (multicollinearity)
- Features that separate classes well

### Next Steps:
- Handle missing values based on patterns observed
- Address class imbalance if needed
- Consider feature engineering based on distributions
- Handle outliers appropriately
- Remove or combine highly correlated features